In [2]:
from langchain_community.llms import Ollama
import pandas as pd
from pathlib import Path
llm = Ollama(model="llama3")
def classify_sentiment(text, prompt_template):
       prompt = prompt_template.format(text=text)
       return llm.invoke(prompt)
news = [ "Shares rise as markets pare expectations for a Federal Reserve rate hike.",
    "Gold gains as a weaker dollar reduces expectations of a near-term Fed rate hike.",
    "US and European shares fell as oil prices rose amid renewed Middle East tensions.",
    "European shares tick higher as easing Fed rate-hike expectations lift gold and mining stocks.",
    "The Indian rupee weakened to a two-week low against the US dollar.",
    "Oil prices climb as investors monitor tensions surrounding Iran and the Strait of Hormuz.",
    "Asian shares rise as investors reduce expectations for an imminent US interest-rate hike.",
    "Technology stocks gain as investors continue to assess the outlook for artificial intelligence investment.",
    "Japanese shares come under pressure after weaker-than-expected economic growth.",
    "The dollar slips as softer US economic data reduces expectations of further interest-rate increases.",
    "Silver prices rise alongside gold as investors expect lower US interest rates.",
    "Global equities gain as investors assess weaker economic data and changing Federal Reserve policy expectations."]
templates = {
      "Chain-of-thought": """Analyze the following news step by step.
       Explain briefly why it is positive, negative, or neutral.
       Now classify: {text}
       Answer with Positive, Negative, or Neutral."""
   }
# Apply the selected prompt to all headlines
results = []

for headline in news:
    output = classify_sentiment(headline, templates["Chain-of-thought"]).strip()

    results.append({
        "headline": headline,
        "model_output": output
    })


# Create DataFrame
df = pd.DataFrame(results)

display(df)
# Manual verification of selected predictions
for i in [0, 5, 8, 10]:
    print(f"\nHeadline: {df.loc[i, 'headline']}")
    print(f"Model output: {df.loc[i, 'model_output']}")
# Export results
output_path = Path("../data/processed/week2_sentiment_predictions.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print(f"Results saved to: {output_path}")


# Display individual results
for i, row in enumerate(df.itertuples(index=False), start=1):
    print(f"\n{i}. {row.headline}")
    print(row.model_output)

,headline,model_output
0,Shares rise as markets pare expectations for a...,Step-by-step analysis:\n\n1. The news article ...
1,Gold gains as a weaker dollar reduces expectat...,Analysis:\n\n* The news is stating that gold p...
2,US and European shares fell as oil prices rose...,Here's the analysis:\n\n**Step 1: Identify the...
3,European shares tick higher as easing Fed rate...,Let's break down the news step by step:\n\n* T...
4,The Indian rupee weakened to a two-week low ag...,Analysis:\n\n* The news states that the Indian...
5,Oil prices climb as investors monitor tensions...,Analysis:\n\nThe news article is reporting on ...
6,Asian shares rise as investors reduce expectat...,Here's the analysis:\n\n**Step 1: Identify the...
7,Technology stocks gain as investors continue t...,Here's the step-by-step analysis:\n\n**News:**...
8,Japanese shares come under pressure after weak...,Analysis:\n\n* The news reports that Japanese ...
9,The dollar slips as softer US economic data re...,Let's break it down step by step!\n\n**Step 1:...



Headline: Shares rise as markets pare expectations for a Federal Reserve rate hike.
Model output: Step-by-step analysis:

1. The news article mentions "shares rise", which indicates a positive trend in the stock market.
2. The article further explains that the rise is due to markets paring expectations for a Federal Reserve rate hike, which suggests that investors are becoming less anxious about interest rate changes.
3. Overall, the article presents a positive outlook, as a rise in shares is generally a good sign for the economy.

Classification: Positive

Headline: Oil prices climb as investors monitor tensions surrounding Iran and the Strait of Hormuz.
Model output: Analysis:

The news article is reporting on an increase in oil prices due to tensions surrounding Iran and the Strait of Hormuz. This suggests that there is a sense of uncertainty or instability in the region, which may lead to higher oil prices.

Reasoning:

* The article is neutral in tone, presenting a factual report

In [4]:
from langchain.agents import create_agent
from langchain.tools import tool
from pydantic import BaseModel, Field
from typing import Literal


class WeatherInput(BaseModel):
    """Input for weather queries."""
    
    location: str = Field(
        description="City name or coordinates"
    )
    
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )


class WeatherOutput(BaseModel):
    description: str = Field(
        description="A description of the weather in the specified city"
    )


@tool(args_schema=WeatherInput)
def get_weather(
    location: str,
    units: str = "celsius",
    include_forecast: bool = False
) -> WeatherOutput:
    """Get current weather and optional forecast."""
    
    temp = 22 if units == "celsius" else 72
    
    result = (
        f"Current weather in {location}: "
        f"{temp} degrees {units[0].upper()}"
    )
    
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    
    return WeatherOutput(description=result)


agent = create_agent(
    model="ollama:devstral-small-2",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)


result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What's the weather in Alice, Eastern Cape, South Africa?"
        }
    ]
})


print(result["messages"][-1].content)

The current weather in Alice, Eastern Cape, South Africa is 22 degrees Celsius.
